# 7 · Data Ingestion — landing raw data in **Bronze**

Ingestion is the first job of a pipeline: get raw data *into* the lakehouse,
reliably and incrementally. In the Medallion architecture the landing spot is the
**Bronze** layer — raw data stored as-is (plus a little metadata), the complete
historical record you can always reprocess from.

This notebook uses the **BrewBox** raw files you created in notebook 6
(`orders/` and `events/` JSON in the `brewbox.landing` Volume) and loads them into
**Bronze Delta tables** three ways:

1. **Batch read** — `spark.read` (simple, re-reads everything each run).
2. **Auto Loader** — `cloudFiles` (incremental: only *new* files, with schema
   evolution). The modern default.
3. **`COPY INTO`** — a SQL, idempotent batch loader.

> Prerequisite: run notebook **6** first so the `brewbox` schema, the `landing`
> Volume, and the raw files exist.

In [ ]:
# Bootstrap: spark + the shared dataset location (re-derived so this notebook is standalone)
try:
    spark
except NameError:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()
from pyspark.sql import functions as F

CATALOG = spark.sql("SELECT current_catalog()").first()[0]
LANDING = f"/Volumes/{CATALOG}/brewbox/landing"
spark.sql("USE SCHEMA brewbox")
print("catalog:", CATALOG, "| landing:", LANDING)

## 1 · Batch read — the simplest ingestion

`spark.read` loads files into a DataFrame. You pick the **format** and pass
**options** (e.g. `multiLine`, `header`, `inferSchema`). Reading is lazy; an
action (`show`, `count`, `write`) triggers it.

In [ ]:
raw_orders = (spark.read
    .format("json")
    .load(f"{LANDING}/orders"))

raw_orders.printSchema()
print("rows:", raw_orders.count())
raw_orders.show(5)

This works, but a **plain batch read reprocesses every file every run**. If new
`orders` files arrive daily, you'd re-read the whole history each time. That's
what Auto Loader fixes.

## 2 · Auto Loader — incremental file ingestion ⭐

**Auto Loader** (`format("cloudFiles")`) incrementally processes **only new
files** as they land, remembering what it has already seen via a **checkpoint**.
It also **infers and evolves the schema** automatically. It's the standard way to
ingest files on Databricks.

Key options:

- `cloudFiles.format` — the file format (`json`, `csv`, `parquet`…).
- `cloudFiles.schemaLocation` — where Auto Loader stores the inferred/evolving schema.
- `checkpointLocation` (on the write) — where it records which files are done.
- `.trigger(availableNow=True)` — process all data available *right now*, then
  stop (batch-style). Omit it for a always-on stream.

We add ingestion metadata (`_ingested_at`, `_source_file`) — a Bronze best
practice — and write to `brewbox.orders_bronze`.

In [ ]:
schema_loc = f"{LANDING}/_schemas/orders"
checkpoint = f"{LANDING}/_checkpoints/orders_bronze"

stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_loc)
    .load(f"{LANDING}/orders")
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path")))

query = (stream.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)          # process what's here now, then stop
    .toTable("brewbox.orders_bronze"))

query.awaitTermination()                 # wait for the batch to finish
print("Bronze rows:", spark.table("brewbox.orders_bronze").count())
spark.table("brewbox.orders_bronze").select(
    "order_id","status","amount","_ingested_at","_source_file").show(5, truncate=False)

**Why this is powerful:** run the same cell again and Auto Loader ingests **0 new
rows** — it remembers the files it already processed via the checkpoint. Drop a
new `orders` file into the Volume and *only that file* is picked up. That's
incremental, exactly-once ingestion with almost no code.

In [ ]:
# Re-run the same Auto Loader batch: no new files -> nothing re-ingested (idempotent)
q2 = (spark.readStream.format("cloudFiles")
    .option("cloudFiles.format","json")
    .option("cloudFiles.schemaLocation", schema_loc)
    .load(f"{LANDING}/orders")
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .writeStream.format("delta")
    .option("checkpointLocation", checkpoint)
    .trigger(availableNow=True)
    .toTable("brewbox.orders_bronze"))
q2.awaitTermination()
print("Bronze rows after re-run (unchanged):", spark.table("brewbox.orders_bronze").count())

## 3 · Schema inference, evolution & rescued data

Semi-structured sources drift — a new field appears, a type changes. Auto Loader
handles this:

- It **infers** the schema and stores it in `schemaLocation`.
- With `cloudFiles.schemaEvolutionMode` (default `addNewColumns`), new columns are
  added automatically and the stream restarts to pick them up.
- Anything that doesn't match lands in a **`_rescued_data`** column instead of
  being dropped — so you never silently lose data.

You can also give a **schema hint** to lock down types you care about:

```python
.option("cloudFiles.schemaHints", "amount double, order_id long")
```

## 4 · `COPY INTO` — idempotent SQL ingestion

`COPY INTO` is a SQL command that loads files into an existing Delta table and
**remembers which files it already loaded** — so re-running it is safe
(idempotent). Great for simpler batch loads or when you prefer pure SQL. Let's
load the `events` files this way.

In [ ]:
# COPY INTO needs a target table to exist first
spark.sql("""
    CREATE TABLE IF NOT EXISTS brewbox.events_bronze (
        event_id BIGINT, customer_id BIGINT, event_type STRING, ts STRING,
        _ingested_at TIMESTAMP
    )
""")

spark.sql(f"""
    COPY INTO brewbox.events_bronze
    FROM (SELECT *, current_timestamp() AS _ingested_at FROM '{LANDING}/events')
    FILEFORMAT = JSON
    FORMAT_OPTIONS ('inferSchema' = 'true')
    COPY_OPTIONS  ('mergeSchema' = 'true')
""")

print("events_bronze rows:", spark.table("brewbox.events_bronze").count())
spark.table("brewbox.events_bronze").show(5)

Run the `COPY INTO` cell again → **0 new rows**: like Auto Loader, it tracks
already-loaded files. **Auto Loader vs COPY INTO:** Auto Loader scales to millions
of files and supports streaming + schema evolution (use it for continuous/large
ingestion); `COPY INTO` is simpler SQL for periodic batch loads.

## 5 · Ingestion patterns & good habits

- **Land raw, transform later.** Bronze keeps data close to source — light typing,
  plus metadata (`_ingested_at`, source file). Cleaning happens in Silver (nb 9).
- **Make it incremental & idempotent.** Auto Loader / `COPY INTO` both process
  each file once, so re-runs are safe.
- **Keep the raw files.** They're your safety net — you can always rebuild Bronze.
- **Where ingestion comes from in real life:** cloud storage (S3/ADLS/GCS),
  databases (via **Lakeflow Connect** / Fivetran / Debezium CDC), and streams
  (**Kafka**, Event Hubs, Kinesis) — but the landing-then-Bronze pattern is the
  same.

## 6 · Exercises

**Exercise 1 —** Read the raw `events` files with a plain batch `spark.read` and
count how many events of each `event_type` there are.

In [ ]:
# Your turn (Exercise 1):

In [ ]:
# ✅ Solution 1
(spark.read.format("json").load(f"{LANDING}/events")
    .groupBy("event_type").count().orderBy(F.desc("count")).show())

**Exercise 2 —** Show the distinct source files that fed `brewbox.orders_bronze`
(hint: the `_source_file` column), and the row count per file.

In [ ]:
# Your turn (Exercise 2):

In [ ]:
# ✅ Solution 2
(spark.table("brewbox.orders_bronze")
    .groupBy("_source_file").count().orderBy("_source_file").show(truncate=False))

**Exercise 3 —** How many rows are in `events_bronze`, and what is the earliest
and latest `ts`? (Use SQL via `spark.sql`.)

In [ ]:
# Your turn (Exercise 3):

In [ ]:
# ✅ Solution 3
spark.sql("""
    SELECT count(*) AS n, min(ts) AS earliest, max(ts) AS latest
    FROM brewbox.events_bronze
""").show(truncate=False)

## 7 · Recap & what's next

You ingested raw BrewBox files into **Bronze** Delta tables three ways —
batch read, **Auto Loader** (incremental, schema-evolving, the default), and
**`COPY INTO`** (idempotent SQL). You now have `brewbox.orders_bronze` and
`brewbox.events_bronze`.

**Next → `8` Delta Lake deep dive:** the storage format under every table here —
`MERGE`/SCD, `OPTIMIZE`, time travel, schema evolution, and more — before we clean
Bronze into Silver in notebook 9. 🚀